# Module 6 lab: measure and protect SQLite work

Query counts and plan output are evidence; no timing threshold is used.

## Objectives and predictions

Predict 1: N+1 queries for three parents? Predict 2: will a join preserve children? Predict 3: what should a stale version update return?

In [ ]:
import sqlite3
conn=sqlite3.connect(":memory:")
conn.executescript("CREATE TABLE parents(id INTEGER PRIMARY KEY,name TEXT); CREATE TABLE children(id INTEGER PRIMARY KEY,parent_id INTEGER,name TEXT); INSERT INTO parents VALUES(1,'A'),(2,'B'),(3,'C'); INSERT INTO children VALUES(1,1,'a1'),(2,1,'a2'),(3,2,'b1'),(4,3,'c1'); CREATE TABLE items(id INTEGER PRIMARY KEY,state TEXT,version INTEGER NOT NULL); INSERT INTO items VALUES(1,'open',1);")
class Counter:
 def __init__(self,c): self.c,self.count=c,0
 def execute(self,s,p=()): self.count+=1; return self.c.execute(s,p)
def n1(db):
 out=[]; ps=db.execute("SELECT id,name FROM parents ORDER BY id").fetchall()
 for p in ps:
  kids=db.execute("SELECT name FROM children WHERE parent_id=? ORDER BY id",(p[0],)).fetchall(); out.append((p[1],[x[0] for x in kids]))
 return out
def join(db):
 rows=db.execute("SELECT p.name,c.name FROM parents p LEFT JOIN children c ON c.parent_id=p.id ORDER BY p.id,c.id").fetchall(); out={}
 for p,c in rows: out.setdefault(p,[]); out[p].append(c) if c else None
 return sorted(out.items())
a=Counter(conn); b=Counter(conn); assert n1(a)==join(b); assert a.count==4 and b.count==1

## Prediction answers

1. Three parents cause four queries in N+1: one parent query plus one per parent. 2. A join should preserve the same parent/child rows when its join and ordering are correct. 3. A stale version update should affect zero rows and report an explicit conflict.

In [ ]:
before=conn.execute("EXPLAIN QUERY PLAN SELECT name FROM children WHERE parent_id=?",(1,)).fetchall()
conn.execute("CREATE INDEX idx_children_parent ON children(parent_id)")
after=conn.execute("EXPLAIN QUERY PLAN SELECT name FROM children WHERE parent_id=?",(1,)).fetchall()
print(before); print(after); assert conn.execute("SELECT name FROM children WHERE parent_id=? ORDER BY id",(1,)).fetchall()==[("a1",),("a2",)]

## Optimistic concurrency and rollback

Two snapshots at version 1 should produce one winner and one conflict.

In [ ]:
first=conn.execute("SELECT state,version FROM items WHERE id=1").fetchone(); second=conn.execute("SELECT state,version FROM items WHERE id=1").fetchone()
def complete(snap):
 cur=conn.execute("UPDATE items SET state=?,version=version+1 WHERE id=? AND version=?",("done",1,snap[1])); conn.commit(); return cur.rowcount
assert complete(first)==1 and complete(second)==0; assert conn.execute("SELECT state,version FROM items").fetchone()==("done",2)
conn.execute("BEGIN"); conn.execute("UPDATE items SET state='broken' WHERE id=1"); conn.rollback(); assert conn.execute("SELECT state FROM items").fetchone()[0]=="done"
ai="CREATE INDEX on every column; pool_size=1000"; assert "pool_size" in ai
c=Counter(conn); join(c); assert c.count==1

## Independent challenge and answers

Add keyset pagination with id > last_id and LIMIT, preserving id order. Answers: N+1 grows one query per parent; indexes cost storage/write work; a stale version affects zero rows; exceptions require rollback and connection release; isolated interventions preserve attribution. This does not prove production capacity or all isolation modes.

Evidence: fixture, counts, plans, one intervention, equivalence, winner/conflict, rollback, pool assumptions, AI review, clean run.

## Baseline reproduction: slow path
Run the N+1 function with three parents and count every query. This is the baseline evidence; do not judge speed from one noisy millisecond value.

In [ ]:
baseline_counter=Counter(conn); baseline_rows=n1(baseline_counter)
assert baseline_counter.count==4 and len(baseline_rows)==3
print("baseline query count:",baseline_counter.count)

## Pre-edit hypothesis
Write two separate hypotheses: (1) a join will reduce query count without changing rows; (2) a version predicate will turn a stale write into an explicit conflict.

In [ ]:
performance_hypothesis="one join should reduce N+1 query count while preserving result order"
concurrency_hypothesis="version in WHERE should produce one winner and one zero-row conflict"
assert "query count" in performance_hypothesis and "zero-row" in concurrency_hypothesis

## Incremental guided implementation: one intervention
First capture the query plan. Then add one index only for the observed child lookup. Do not combine this with pool-size or unrelated changes.

In [ ]:
assert any("SCAN" in " ".join(map(str,row)) for row in before)
assert any("INDEX" in " ".join(map(str,row)).upper() for row in after)

In [ ]:
joined_counter=Counter(conn); joined_rows=join(joined_counter)
assert joined_counter.count==1 and joined_rows==baseline_rows
print("one attributable intervention: join; rows preserved")

## Transaction reasoning
A transaction is a boundary around related changes. A failed change must roll back; a conditional update uses the version read earlier to detect a stale snapshot.

In [ ]:
conn.execute("INSERT INTO items VALUES (2,'open',1)"); conn.commit()
snap_a=conn.execute("SELECT state,version FROM items WHERE id=2").fetchone(); snap_b=conn.execute("SELECT state,version FROM items WHERE id=2").fetchone()
def versioned_update(item_id,snap):
 cur=conn.execute("UPDATE items SET state='done',version=version+1 WHERE id=? AND version=?",(item_id,snap[1])); conn.commit(); return cur.rowcount
assert versioned_update(2,snap_a)==1 and versioned_update(2,snap_b)==0

## Positive, negative, and failure checks
Positive means equivalent results after the intervention. Negative means a stale writer loses explicitly. Failure means rollback restores the prior state and a bounded retry decision is recorded.

In [ ]:
conn.execute("BEGIN"); conn.execute("UPDATE items SET state='corrupt' WHERE id=2"); conn.rollback()
assert conn.execute("SELECT state FROM items WHERE id=2").fetchone()[0]=="done"
retry_plan={"max_attempts":2,"replay_safe":True}
assert retry_plan["max_attempts"]==2 and retry_plan["replay_safe"]

## AI-style/broken-code critique
The generated advice adds every index and increases a pool to hide query cost. It also retries writes without checking whether a side effect already happened.

In [ ]:
broken_perf="CREATE INDEX on every column; pool=1000; retry update forever"
assert "every column" in broken_perf and "forever" in broken_perf
print("Reject unmeasured indexes, pool camouflage, and unbounded unsafe retry.")

## Guided TODO: attempt
Implement a bounded keyset page using id > last_id and a limit. This preserves deterministic order and avoids a time-based assertion.

In [ ]:
def todo_keyset(last_id,limit):
 return conn.execute("SELECT id,name FROM children WHERE id>? ORDER BY id LIMIT ?",(last_id,limit)).fetchall()
assert todo_keyset(1,2)==[(2,"a2"),(3,"b1")]

## Reference solution
The reference clamps the limit, orders by the cursor column, and uses parameters.

In [ ]:
def reference_keyset(last_id,limit):
 limit=max(1,min(limit,50))
 return conn.execute("SELECT id,name FROM children WHERE id>? ORDER BY id LIMIT ?",(last_id,limit)).fetchall()
assert reference_keyset(0,999)[0]==(1,"a1") and len(reference_keyset(0,999))<=4

## Independent challenge
Explain when optimistic concurrency is preferable to a lock and when a short lock may be simpler. Keep fixture size and one intervention unchanged.

In [ ]:
decision={"strategy":"optimistic version","reason":"short local edits can report conflicts without holding a lock"}
assert decision["strategy"]=="optimistic version" and "conflicts" in decision["reason"]

## Exit questions and Answers
N+1 grows one query per parent. An index costs space and write work. EXPLAIN is plan evidence, not universal latency. A version predicate makes stale updates affect zero rows. Rollback and connection release are required after failure. One intervention preserves attribution.

## Evidence handoff
Keep fixture, query counts, before/after plans, separate hypotheses, one join/index intervention, result equivalence, winner/conflict, rollback/retry evidence, pool assumptions, AI review, TODO/reference, challenge decision, and clean run.